# 财政 重构

## 元数据
- 原始路径: g:\我的云端硬盘\workspace\projects\github\work_station\2025\财政
- 创建时间: 2026-02-15
- 任务ID: T22

## 1. 项目概述

### 1.1 功能描述

财政数据整合导入项目，用于读取、清洗和导入中国财政数据到数据库。支持多年度、多维度的财政数据管理。

**主要功能**：
- 财政数据读取（Excel格式）
- 数据清洗和转换
- 地域信息解析（省级/地市级）
- 数据库表创建和数据导入
- 多年份批量处理

### 1.2 数据源

- **数据文件**: `data/17-24chinafiscaldata.xlsx`
- **数据范围**: 2017-2024年全国省级和地市级财政数据
- **数据格式**: Excel文件，每年一个工作表

### 1.3 输出结果

- **数据库表**: `china_fiscal_data_2017_2024`
- **记录数**: 取决于Excel数据量
- **覆盖范围**: 2017-2024年全国省级和地市级财政数据

## 2. 环境配置

### 2.1 导入依赖

In [ ]:
import pandas as pd
import numpy as np
import logging
import os
from datetime import datetime
from typing import Dict, List, Optional, Tuple
from pathlib import Path

# 数据库相关
from sqlalchemy import create_engine, text
from sqlalchemy.types import DECIMAL, VARCHAR, INT, DATE, TIMESTAMP, TEXT
import pymysql

# 配置导入
from config import *

# 设置日志
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

print("依赖导入完成")

### 2.2 配置参数

In [ ]:
# 显示配置信息
print(f"项目根目录: {PROJECT_ROOT}")
print(f"数据目录: {DATA_DIR}")
print(f"输出目录: {OUTPUT_DIR}")
print(f"数据文件路径: {EXCEL_FILE_PATH}")
print(f"目标表名: {TABLE_NAME}")

## 3. 数据获取

### 3.1 数据库连接

In [ ]:
def get_database_engine():
    """创建数据库连接引擎"""
    # 使用环境变量获取敏感信息
    db_host = os.environ.get('DB_HOST', DB_HOST)
    db_port = os.environ.get('DB_PORT', str(DB_PORT))
    db_user = os.environ.get('DB_USER', DB_USER)
    db_password = os.environ.get('DB_PASSWORD', DB_PASSWORD)
    db_name = os.environ.get('DB_NAME', DB_NAME)
    
    connection_string = f"mysql+pymysql://{db_user}:{db_password}@{db_host}:{db_port}/{db_name}?charset=utf8mb4"
    
    engine = create_engine(
        connection_string,
        pool_recycle=3600,
        pool_size=10,
        max_overflow=20,
        echo=False
    )
    
    return engine

# 创建数据库连接
engine = get_database_engine()

# 测试连接
try:
    with engine.connect() as conn:
        result = conn.execute(text("SELECT 1"))
        print("数据库连接成功")
except Exception as e:
    print(f"数据库连接失败: {e}")

### 3.2 数据读取

In [ ]:
# 检查数据文件是否存在
if os.path.exists(EXCEL_FILE_PATH):
    # 读取Excel文件
    excel_file = pd.ExcelFile(EXCEL_FILE_PATH)
    sheet_names = excel_file.sheet_names
    
    # 筛选数字年份的工作表
    year_sheets = [name for name in sheet_names if name.isdigit()]
    
    print(f"发现工作表: {sheet_names}")
    print(f"年份工作表: {year_sheets}")
    
    # 读取第一个年份的数据作为预览
    if year_sheets:
        df_preview = pd.read_excel(EXCEL_FILE_PATH, sheet_name=year_sheets[0])
        print(f"\n{year_sheets[0]}年数据预览:")
        print(f"数据形状: {df_preview.shape}")
        print(f"列名: {list(df_preview.columns)[:10]}...")
        display(df_preview.head(3))
else:
    print(f"数据文件不存在: {EXCEL_FILE_PATH}")

## 4. 数据处理

### 4.1 数据清洗

In [ ]:
def clean_data(df: pd.DataFrame) -> pd.DataFrame:
    """
    数据清洗
    
    Args:
        df: 原始数据框
        
    Returns:
        清洗后的数据框
    """
    # 替换特殊值为NaN
    df = df.replace('--', np.nan)
    df = df.replace('', np.nan)
    
    # 删除全为空的行
    df = df.dropna(how='all')
    
    return df

# 测试数据清洗
if 'df_preview' in locals():
    df_cleaned = clean_data(df_preview.copy())
    print(f"清洗前: {len(df_preview)} 行")
    print(f"清洗后: {len(df_cleaned)} 行")

### 4.2 数据转换

In [ ]:
# 省级行政区列表
PROVINCES = [
    '北京', '天津', '上海', '重庆', '河北', '山西', '辽宁', '吉林', '黑龙江',
    '江苏', '浙江', '安徽', '福建', '江西', '山东', '河南', '湖北', '湖南',
    '广东', '海南', '四川', '贵州', '云南', '陕西', '甘肃', '青海', '台湾',
    '内蒙古', '广西', '西藏', '宁夏', '新疆', '香港', '澳门', '中国'
]

def parse_region_info(region_name: str, yy_region: Optional[str] = None) -> Tuple[str, str, str]:
    """
    解析地域信息，区分省、市和地区级别
    
    Args:
        region_name: 地域名称
        yy_region: YY地域信息（可选）
        
    Returns:
        Tuple[province, city, level]: 省、市、地区级别
    """
    if pd.isna(region_name) or region_name == '':
        return '', '', ''
    
    # 清理地域名称
    region_name = str(region_name).strip()
    
    # 检查是否为省级
    for province in PROVINCES:
        if region_name == province:
            return province, '', '省级'
        elif region_name.startswith(province) and len(region_name) > len(province):
            return province, region_name, '地市级'
    
    # 默认为地市级
    return '', region_name, '地市级'

def convert_data_types(df: pd.DataFrame) -> pd.DataFrame:
    """
    转换数据类型
    
    Args:
        df: 原始数据框
        
    Returns:
        转换后的数据框
    """
    # 定义数值列
    numeric_columns = [
        '负债率(%)', '负债率(宽口径)(%)', '债务率(%)', '债务率(宽口径)(%)',
        '城投债余额(亿)', '地方政府债余额(亿)', '地方政府债限额(亿)',
        '城投有息负债(亿)', '非标融资余额(亿)', '非标融资余额/城投有息负债(%)',
        '人民币: 各项存款余额/城投有息负债(%)', '城投有息负债: 三年复合增速(%)',
        'GDP(亿)', 'GDP增速(%)', '人均GDP(元)', '房地产投资(亿)',
        '城镇居民人均可支配收入(元)', '商品房销售面积(万平方米)',
        '商品房销售金额(亿)', '城镇居民人均消费性支出(元)',
        '农村居民人均消费性支出(元)', '工业增加量(亿)',
        '本外币: 各项存款余额(亿)', '本外币: 各项贷款余额(亿)',
        '地方政府综合财力(亿)', '财政自给率(%)', '一般公共预算收入(亿)',
        '一般公共预算支出(亿)', '税收收入(亿)', '税收收入占比(%)',
        '政府性基金收入(亿)', '政府性基金支出(亿)',
        '国有土地权出让收入(亿)', '国有土地权出让收入: 三年复合增速(%)',
        '国有土地权出让收入/一般公共预算收入(%)', '常住人口(万)'
    ]
    
    # 转换数值列
    for col in numeric_columns:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    
    return df

# 测试地域解析
test_regions = ['北京', '江苏', '南京市', '苏州市', '深圳']
for region in test_regions:
    province, city, level = parse_region_info(region)
    print(f"{region} -> 省: {province}, 市: {city}, 级别: {level}")

## 5. 核心逻辑

### 5.1 主函数实现

In [ ]:
def process_year_data(year: int, df: pd.DataFrame) -> pd.DataFrame:
    """
    处理单个年份的数据
    
    Args:
        year: 年份
        df: 年份数据框
        
    Returns:
        处理后的数据框
    """
    logger.info(f"处理{year}年数据，共{len(df)}行")
    
    # 数据清洗
    df = clean_data(df)
    
    # 转换数据类型
    df = convert_data_types(df)
    
    # 添加年份列
    df['year'] = year
    
    # 添加dt列（每年的最后一天）
    dt_date = datetime(year, 12, 31).date()
    df['dt'] = dt_date
    
    # 解析地域信息
    region_info = df.apply(
        lambda row: parse_region_info(row['地域'], row.get('YY地域', None)),
        axis=1
    )
    df['province'] = region_info.apply(lambda x: x[0])
    df['city'] = region_info.apply(lambda x: x[1])
    df['region_level'] = region_info.apply(lambda x: x[2])
    
    # 重排序列
    desired_columns = [
        'dt', 'year', '地域', 'province', 'city', 'region_level', 'YY地域'
    ] + [col for col in df.columns if col not in [
        'dt', 'year', '地域', 'province', 'city', 'region_level', 'YY地域'
    ]]
    
    df = df[desired_columns]
    
    logger.info(f"{year}年数据处理完成，有效数据{len(df)}行")
    
    return df

print("process_year_data 函数定义完成")

### 5.2 辅助函数 - 数据库表创建

In [ ]:
def create_fiscal_table(engine, table_name: str):
    """
    创建财政数据表
    
    Args:
        engine: SQLAlchemy引擎
        table_name: 表名
    """
    create_table_sql = f"""
    CREATE TABLE IF NOT EXISTS {table_name} (
        id INT AUTO_INCREMENT PRIMARY KEY,
        dt DATE NOT NULL COMMENT '日期（每年最后一天）',
        year INT NOT NULL COMMENT '年份',
        region_name VARCHAR(100) COMMENT '地域名称',
        province VARCHAR(50) COMMENT '省份',
        city VARCHAR(50) COMMENT '城市',
        region_level ENUM('省级', '地市级') COMMENT '地区级别',
        yy_region TEXT COMMENT 'YY地域描述',
        debt_ratio DECIMAL(10,4) COMMENT '负债率(%)',
        debt_ratio_wide DECIMAL(10,4) COMMENT '负债率(宽口径)(%)',
        debt_to_gdp_ratio DECIMAL(10,4) COMMENT '债务率(%)',
        debt_to_gdp_ratio_wide DECIMAL(10,4) COMMENT '债务率(宽口径)(%)',
        urban_investment_bond_balance DECIMAL(20,2) COMMENT '城投债余额(亿)',
        local_government_debt_balance DECIMAL(20,2) COMMENT '地方政府债余额(亿)',
        local_government_debt_limit DECIMAL(20,2) COMMENT '地方政府债限额(亿)',
        urban_investment_interest_debt DECIMAL(20,2) COMMENT '城投有息负债(亿)',
        non_standard_financing_balance DECIMAL(20,2) COMMENT '非标融资余额(亿)',
        non_standard_financing_ratio DECIMAL(10,4) COMMENT '非标融资余额/城投有息负债(%)',
        deposit_to_debt_ratio DECIMAL(10,4) COMMENT '人民币: 各项存款余额/城投有息负债(%)',
        urban_investment_debt_growth_3y DECIMAL(10,4) COMMENT '城投有息负债: 三年复合增速(%)',
        gdp DECIMAL(20,2) COMMENT 'GDP(亿)',
        gdp_growth_rate DECIMAL(10,4) COMMENT 'GDP增速(%)',
        gdp_per_capita DECIMAL(20,2) COMMENT '人均GDP(元)',
        real_estate_investment DECIMAL(20,2) COMMENT '房地产投资(亿)',
        urban_disposable_income DECIMAL(20,2) COMMENT '城镇居民人均可支配收入(元)',
        commercial_housing_area DECIMAL(20,2) COMMENT '商品房销售面积(万平方米)',
        commercial_housing_amount DECIMAL(20,2) COMMENT '商品房销售金额(亿)',
        urban_consumption_expenditure DECIMAL(20,2) COMMENT '城镇居民人均消费性支出(元)',
        rural_consumption_expenditure DECIMAL(20,2) COMMENT '农村居民人均消费性支出(元)',
        industrial_added_value DECIMAL(20,2) COMMENT '工业增加量(亿)',
        total_deposits DECIMAL(20,2) COMMENT '本外币: 各项存款余额(亿)',
        total_loans DECIMAL(20,2) COMMENT '本外币: 各项贷款余额(亿)',
        local_government_fiscal_power DECIMAL(20,2) COMMENT '地方政府综合财力(亿)',
        fiscal_self_sufficiency_ratio DECIMAL(10,4) COMMENT '财政自给率(%)',
        general_public_budget_revenue DECIMAL(20,2) COMMENT '一般公共预算收入(亿)',
        general_public_budget_expenditure DECIMAL(20,2) COMMENT '一般公共预算支出(亿)',
        tax_revenue DECIMAL(20,2) COMMENT '税收收入(亿)',
        tax_revenue_ratio DECIMAL(10,4) COMMENT '税收收入占比(%)',
        government_fund_revenue DECIMAL(20,2) COMMENT '政府性基金收入(亿)',
        government_fund_expenditure DECIMAL(20,2) COMMENT '政府性基金支出(亿)',
        state_owned_land_transfer_revenue DECIMAL(20,2) COMMENT '国有土地权出让收入(亿)',
        land_transfer_revenue_growth_3y DECIMAL(10,4) COMMENT '国有土地权出让收入: 三年复合增速(%)',
        land_transfer_to_budget_ratio DECIMAL(10,4) COMMENT '国有土地权出让收入/一般公共预算收入(%)',
        resident_population DECIMAL(20,2) COMMENT '常住人口(万)',
        created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
        updated_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP ON UPDATE CURRENT_TIMESTAMP,
        INDEX idx_dt (dt),
        INDEX idx_year (year),
        INDEX idx_region (region_name),
        INDEX idx_province (province),
        INDEX idx_city (city),
        INDEX idx_region_level (region_level)
    ) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COLLATE=utf8mb4_unicode_ci
    COMMENT='中国财政数据2017-2024'
    """
    
    try:
        with engine.connect() as conn:
            conn.execute(text(create_table_sql))
            conn.commit()
        logger.info(f"表 {table_name} 创建成功")
    except Exception as e:
        logger.error(f"创建表失败: {e}")
        raise

print("create_fiscal_table 函数定义完成")

### 5.3 辅助函数 - 数据导入

In [ ]:
# 列名映射：中文 -> 英文字段名
COLUMN_MAPPING = {
    '地域': 'region_name',
    'YY地域': 'yy_region',
    '负债率(%)': 'debt_ratio',
    '负债率(宽口径)(%)': 'debt_ratio_wide',
    '债务率(%)': 'debt_to_gdp_ratio',
    '债务率(宽口径)(%)': 'debt_to_gdp_ratio_wide',
    '城投债余额(亿)': 'urban_investment_bond_balance',
    '地方政府债余额(亿)': 'local_government_debt_balance',
    '地方政府债限额(亿)': 'local_government_debt_limit',
    '城投有息负债(亿)': 'urban_investment_interest_debt',
    '非标融资余额(亿)': 'non_standard_financing_balance',
    '非标融资余额/城投有息负债(%)': 'non_standard_financing_ratio',
    '人民币: 各项存款余额/城投有息负债(%)': 'deposit_to_debt_ratio',
    '城投有息负债: 三年复合增速(%)': 'urban_investment_debt_growth_3y',
    'GDP(亿)': 'gdp',
    'GDP增速(%)': 'gdp_growth_rate',
    '人均GDP(元)': 'gdp_per_capita',
    '房地产投资(亿)': 'real_estate_investment',
    '城镇居民人均可支配收入(元)': 'urban_disposable_income',
    '商品房销售面积(万平方米)': 'commercial_housing_area',
    '商品房销售金额(亿)': 'commercial_housing_amount',
    '城镇居民人均消费性支出(元)': 'urban_consumption_expenditure',
    '农村居民人均消费性支出(元)': 'rural_consumption_expenditure',
    '工业增加量(亿)': 'industrial_added_value',
    '本外币: 各项存款余额(亿)': 'total_deposits',
    '本外币: 各项贷款余额(亿)': 'total_loans',
    '地方政府综合财力(亿)': 'local_government_fiscal_power',
    '财政自给率(%)': 'fiscal_self_sufficiency_ratio',
    '一般公共预算收入(亿)': 'general_public_budget_revenue',
    '一般公共预算支出(亿)': 'general_public_budget_expenditure',
    '税收收入(亿)': 'tax_revenue',
    '税收收入占比(%)': 'tax_revenue_ratio',
    '政府性基金收入(亿)': 'government_fund_revenue',
    '政府性基金支出(亿)': 'government_fund_expenditure',
    '国有土地权出让收入(亿)': 'state_owned_land_transfer_revenue',
    '国有土地权出让收入: 三年复合增速(%)': 'land_transfer_revenue_growth_3y',
    '国有土地权出让收入/一般公共预算收入(%)': 'land_transfer_to_budget_ratio',
    '常住人口(万)': 'resident_population'
}

def import_data_to_db(engine, df: pd.DataFrame, table_name: str, chunksize: int = 1000):
    """
    导入数据到数据库
    
    Args:
        engine: SQLAlchemy引擎
        df: 要导入的数据框
        table_name: 目标表名
        chunksize: 批量插入大小
    """
    if df.empty:
        logger.warning("没有数据需要导入")
        return
    
    # 重命名列
    df_import = df.rename(columns=COLUMN_MAPPING)
    
    # 只选择映射成功的列
    valid_columns = [col for col in df_import.columns if col in COLUMN_MAPPING.values() or col in [
        'dt', 'year', 'province', 'city', 'region_level'
    ]]
    df_import = df_import[valid_columns]
    
    # 批量导入数据
    try:
        df_import.to_sql(
            table_name,
            con=engine,
            if_exists='append',
            index=False,
            chunksize=chunksize
        )
        logger.info(f"数据导入完成，共导入{len(df_import)}条记录")
    except Exception as e:
        logger.error(f"数据导入失败: {e}")
        raise

print("import_data_to_db 函数定义完成")

## 6. 执行与测试

### 6.1 执行主流程

In [ ]:
def main():
    """主函数 - 执行完整的数据导入流程"""
    logger.info("开始执行财政数据导入流程")
    
    try:
        # 1. 创建数据库表
        logger.info("步骤1: 创建数据库表")
        create_fiscal_table(engine, TABLE_NAME)
        
        # 2. 读取Excel文件
        logger.info(f"步骤2: 读取Excel文件 - {EXCEL_FILE_PATH}")
        excel_file = pd.ExcelFile(EXCEL_FILE_PATH)
        sheet_names = [name for name in excel_file.sheet_names if name.isdigit()]
        logger.info(f"发现年份工作表: {sheet_names}")
        
        # 3. 处理每个年份的数据
        logger.info("步骤3: 处理各年份数据")
        all_data = []
        for year_str in sheet_names:
            year = int(year_str)
            df = pd.read_excel(EXCEL_FILE_PATH, sheet_name=year_str)
            processed_df = process_year_data(year, df)
            all_data.append(processed_df)
        
        # 4. 合并所有数据
        logger.info("步骤4: 合并所有年份数据")
        if all_data:
            combined_df = pd.concat(all_data, ignore_index=True)
            logger.info(f"数据合并完成，总记录数: {len(combined_df)}")
            
            # 5. 导入数据库
            logger.info("步骤5: 导入数据到数据库")
            import_data_to_db(engine, combined_df, TABLE_NAME)
            
            # 6. 显示统计信息
            logger.info("="*50)
            logger.info("数据导入统计:")
            logger.info(f"总记录数: {len(combined_df)}")
            logger.info(f"年份范围: {combined_df['year'].min()} - {combined_df['year'].max()}")
            logger.info(f"省级记录数: {len(combined_df[combined_df['region_level'] == '省级'])}")
            logger.info(f"地市级记录数: {len(combined_df[combined_df['region_level'] == '地市级'])}")
            logger.info("="*50)
        else:
            logger.warning("没有找到有效的数据")
            
    except Exception as e:
        logger.error(f"程序执行失败: {e}")
        raise

print("main 函数定义完成")

In [ ]:
# 执行主流程（取消注释以下代码以执行）
# main()

print("提示: 取消注释 main() 以执行完整的数据导入流程")

### 6.2 结果验证

In [ ]:
def verify_data(engine, table_name: str):
    """
    验证导入的数据
    
    Args:
        engine: SQLAlchemy引擎
        table_name: 表名
    """
    try:
        # 查询数据统计
        query = f"""
        SELECT 
            year, 
            region_level, 
            COUNT(*) as count 
        FROM {table_name} 
        GROUP BY year, region_level
        ORDER BY year, region_level
        """
        
        with engine.connect() as conn:
            result = pd.read_sql(query, conn)
        
        print("数据验证结果:")
        print("="*50)
        display(result)
        
        # 查询总记录数
        count_query = f"SELECT COUNT(*) as total FROM {table_name}"
        with engine.connect() as conn:
            total = pd.read_sql(count_query, conn)
        print(f"\n总记录数: {total['total'].iloc[0]}")
        
    except Exception as e:
        print(f"验证失败: {e}")

# 执行验证（取消注释以下代码以执行）
# verify_data(engine, TABLE_NAME)

print("verify_data 函数定义完成")

## 7. 工具函数（可复用）

In [ ]:
def get_fiscal_data(engine, table_name: str, 
                    years: Optional[List[int]] = None,
                    provinces: Optional[List[str]] = None,
                    region_level: Optional[str] = None) -> pd.DataFrame:
    """
    查询财政数据
    
    Args:
        engine: SQLAlchemy引擎
        table_name: 表名
        years: 年份列表（可选）
        provinces: 省份列表（可选）
        region_level: 地区级别（可选）
        
    Returns:
        查询结果DataFrame
    """
    query = f"SELECT * FROM {table_name} WHERE 1=1"
    
    if years:
        query += f" AND year IN ({','.join(map(str, years))})"
    if provinces:
        query += f" AND province IN ({','.join(repr(p) for p in provinces)})"
    if region_level:
        query += f" AND region_level = '{region_level}'"
    
    with engine.connect() as conn:
        df = pd.read_sql(query, conn)
    
    return df

def export_to_excel(df: pd.DataFrame, output_path: str, sheet_name: str = 'Sheet1'):
    """
    导出数据到Excel文件
    
    Args:
        df: 要导出的DataFrame
        output_path: 输出文件路径
        sheet_name: 工作表名称
    """
    df.to_excel(output_path, sheet_name=sheet_name, index=False)
    logger.info(f"数据已导出到: {output_path}")

def get_data_quality_report(df: pd.DataFrame) -> pd.DataFrame:
    """
    生成数据质量报告
    
    Args:
        df: 要分析的DataFrame
        
    Returns:
        数据质量报告DataFrame
    """
    report = pd.DataFrame({
        '列名': df.columns,
        '非空数': df.notna().sum().values,
        '空值数': df.isna().sum().values,
        '空值率(%)': (df.isna().sum() / len(df) * 100).values,
        '唯一值数': df.nunique().values
    })
    return report

print("工具函数定义完成")

In [ ]:
# 示例：生成数据质量报告
if 'df_preview' in locals():
    quality_report = get_data_quality_report(df_preview)
    print("数据质量报告示例:")
    display(quality_report.head(10))